# 014 — SDOF validation by IDA, and collapse fragility curves

Validates the parameterised SDOF from `013` against the MDOF frames using incremental dynamic
analysis (IDA) with the 22-record FEMA P695 far-field set, then derives collapse fragility
curves for all three model types.

| Notebook | Does |
|---|---|
| `012-mdof_casestudy_analyses` | Build MDOF models + analysis files, design dataset |
| `013-sdof_backbone_parameterisation` | Fit the parameterised SDOF to the MDOF response |
| **014** (this one) | Validate by IDA, produce fragility curves |

**Upstream — `013` must have been run to completion, including both of its run barriers.**
This notebook reads:

- `cfg["proc_data"]["sdof_parameters"]` — `<tag>_sdof_parameters.json`, `participation_factors.json`
- `cfg["proc_data"]["sdof_optimisation"]` — `<tag>_optimised_parameters.json`
- `cfg["proc_data"]["approx_sdof_systems"]` — `<tag>_appx_sdof_{bb,hyst}_parameters.json`
- `<ROOT>/<tag>/modal/modal_properties.json` (from `012`)
- the FEMA P695 records in `E:/gm_records_p695`

**Downstream:** `017-disagg_imls_for_msa_stripes` reads `mdof_fragility_curves.pickle` and
`sdof_fragility_curves.pickle` from `cfg["proc_data"]["sdof_fragility_curves"]`.
`050_setup_msa_runs_for_complete_sites` reuses the SDOF/MDOF model folders.

## ⚠ This notebook contains one run barrier

§1 writes the IDA launchers; §3 onwards reads their results. The IDA is by far the longest
analysis in the chain — expect it to take a long time even across many cores.

In [ ]:
%load_ext autoreload
%autoreload 2

## §0 Setup

In [ ]:
import os
import json
import shutil
import pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
from numpy.polynomial import Polynomial
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TABLEAU_COLORS
from matplotlib.lines import Line2D
from scipy.stats import lognorm
from scipy.optimize import curve_fit
import statsmodels.api as sm

from phd_project.config import config
from phd_project.scripts.standardise_responses import standardise_responses
from standes.analysis.recorders import get_recorder, get_recorders

from phd_project.scripts.templates.copy_templates_to_folders import (
    copy_analysis_config,
    copy_structural_model,
    configure_batch_run_file,
    copy_file,
)

from phd_project.scripts.WP1_ground_motion_set.optimise_sdof_parameters_differential_evolution import (
    validate_optimised_parameters,
)

# Fitting primitives and the *fitted* prediction equations both live in a shared
# module -- single source of truth, also used by 011 and by the SDOF model template.
# See phd_project/scripts/sdof_parameterisation.py
from phd_project.scripts.sdof_parameterisation import (
    linear_model,
    quadratic,
    exponential_shifted,
    linear_constant_model,
    bilinear_piecewise_model,
    calculate_fit_metrics,
    get_approximate_ss_backbone,
    get_approximate_ss_backbone_params,
    get_approximate_br_backbone,
    br_backbone_params,
    get_approximate_hysteretic_hyst_params,
    get_approximate_steel02_hyst_params,
    total_backbone,
)

from fitpo import (
    fit_cbf_backbone,
    fit_envelope,
    fit_piecewise_backbone,
    fit_tetralinear_backbone,
)
from fitpo.fitting import elastic_slope

cfg = config.load_config()

In [ ]:
def _load_pushover_curve(building_folder: Path):
    # load a pushover curve
    po_folder = building_folder / "pushover"
    po_curve_file = po_folder / "po_curve.csv"

    poc = np.loadtxt(po_curve_file, delimiter=",")
    return poc

def _load_cyclic_pushover_curve(building_folder: Path):
    # load a pushover curve
    po_folder = building_folder / "cyclic_pushover"
    po_curve_file = po_folder / "po_curve.csv"

    poc = np.loadtxt(po_curve_file, delimiter=",")
    return poc

In [ ]:
ROOT = cfg["analysis_data"]["dc2_sdof_fitting"]

In [ ]:
# load the building dataset
with open(cfg["proc_data"]["dc2_casestudy_dataset"], "rb") as f:
    bdata_df = pickle.load(f)

bdata_df = bdata_df.set_index("name")
# 012 writes the dataset with a clean RangeIndex, so there is no stray "index" column
# to drop here (older pickles produced by the previous 014 notebook did have one).
bdata_df = bdata_df.drop(columns="index", errors="ignore")
bdata_df

storey_filter = 3
buildings = bdata_df[bdata_df["n_storeys"] == storey_filter].index
# buildings = bdata_df.index

pocs = {}

for b in buildings:
    pocs[b] = _load_pushover_curve(ROOT / b)

### Reload the results carried over from `013`

`gamma` per building is reloaded from the JSON that `013` §1 wrote, rather than recomputed,
so the two notebooks cannot drift apart.

In [ ]:
# participation factors written by 013 section 1
with open(cfg["proc_data"]["sdof_parameters"] / "participation_factors.json") as file:
    gammas = json.load(file)

# the 013 half built `configs_3s` for the optimisation; downstream it is only used to
# size subplot grids, so the building list is all that is actually needed here.
assert set(buildings) <= set(gammas), "missing participation factors -- rerun 013 section 1"

print(f"{len(buildings)} three-storey buildings; gamma range "
      f"{min(gammas[b] for b in buildings):.3f} - {max(gammas[b] for b in buildings):.3f}")

## §1 Build the IDA analysis folders and launchers

Creates IDA inputs for three model types — the MDOF frame, the optimised SDOF and the
approximate (fully-predicted) SDOF — plus modal analyses of the two SDOF systems so their
periods can be compared in §3.

The results for the 3s buildings look good for the cyclic pushovers but we need to see how they look with some time histories as well to see if they are ballpark

Create some files to run IDAs for a single record for each of the 3s MDOF Buildings

In [ ]:
batch = []

for b in buildings:
    # copy files that don't need changing:
    src_proc = cfg["templates"]["ida_process_recorder_roof_drift"]
    dst_proc = ROOT / b / "ida_process_recorders.py"
    copy_file(src_proc, dst_proc)

    src_im = cfg["templates"]["config_im_SA"]
    dst_im = ROOT / b / "config_im.py"
    copy_file(src_im, dst_im)

    src_inj = cfg["templates"]["nltha_injection_update_damping"]
    dst_inj = ROOT / b / "injection_functions.py"
    copy_file(src_inj, dst_inj)

    src_run = cfg["templates"]["run_ida_htf_single_record"]
    dst_run = ROOT / b / "run_ida_htf_single_record.py"
    copy_file(src_run, dst_run)

    # now copy the config 
    src_config = cfg["templates"]["config_ida_htf"]
    dst_config = ROOT / b / "config_ida_htf_120111.py"

    changes = {
        "results_folder_name": "ida_120111",
        "record_filenames": ['fema_p695_120111.json'],
    }

    copy_analysis_config(src_config, dst_config, **changes)
    
    batch.append({
        "script": dst_run,
        "config": [dst_config]
        })
    
batch_run_src = cfg["templates"]["batch_run"]
batch_run_dst = cfg["scripts"]["wp1pt4pt1_batch_run"] / "mdof_3s_idas_120111.py"
configure_batch_run_file(batch_run_src, batch_run_dst, batch)

Create some files to run full IDAs for to calculate the fragility of the MDOF frames

In [ ]:
# most of the files are the same - just need a new run and config file
batch = []

for b in buildings:

    src_run = cfg["templates"]["run_ida_htf_multiple_records"]
    dst_run = ROOT / b / "run_ida_htf_multiple_records.py"
    copy_file(src_run, dst_run)

    # now copy the config 
    src_config = cfg["templates"]["config_ida_htf"]
    dst_config = ROOT / b / "config_ida_htf_femap695_set.py"

    changes = {
        "results_folder_name": "ida_femap695",
        "record_filenames": ['fema_p695_120111.json',
                             'fema_p695_120121.json',
                             'fema_p695_120411.json',
                             'fema_p695_120521.json',
                             'fema_p695_120611.json',
                             'fema_p695_120621.json',
                             'fema_p695_120711.json',
                             'fema_p695_120721.json',
                             'fema_p695_120811.json',
                             'fema_p695_120821.json',
                             'fema_p695_120911.json',
                             'fema_p695_120921.json',
                             'fema_p695_121011.json',
                             'fema_p695_121021.json',
                             'fema_p695_121111.json',
                             'fema_p695_121211.json',
                             'fema_p695_121221.json',
                             'fema_p695_121321.json',
                             'fema_p695_121411.json',
                             'fema_p695_121421.json',
                             'fema_p695_121511.json',
                             'fema_p695_121711.json',
                             ],
    }

    copy_analysis_config(src_config, dst_config, **changes)
    
    batch.append({
        "script": dst_run,
        "config": [dst_config]
        })
    
batch_run_src = cfg["templates"]["batch_run"]
batch_run_dst = cfg["scripts"]["wp1pt4pt1_batch_run"] / "mdof_3s_ida_femaP695.py"
configure_batch_run_file(batch_run_src, batch_run_dst, batch)

Create some files for running a full IDA on the SDOF Systems

In [ ]:
batch = []

for b in buildings:
    # make the target folder
    folder = ROOT / f"{b}_sdof"
    folder.mkdir(parents=True, exist_ok=True)

    # copy files that don't need changing:
    src_proc = cfg["templates"]["ida_process_recorder_x_displacement"]
    dst_proc = folder / "ida_process_recorders.py"
    copy_file(src_proc, dst_proc)

    src_im = cfg["templates"]["config_im_SA"]
    dst_im = folder / "config_im.py"
    copy_file(src_im, dst_im)

    src_inj = cfg["templates"]["nltha_injection_update_damping"]
    dst_inj = folder / "injection_functions.py"
    copy_file(src_inj, dst_inj)

    src_run = cfg["templates"]["run_ida_htf_multiple_records"]
    dst_run = folder / "run_ida_htf_multiple_records.py"
    copy_file(src_run, dst_run)

    # now copy the structural model
    src_model = cfg["templates"]["model_cbf_sdof"]
    dst_model = folder / "structural_model.py"

    fp_sdof_params = cfg["proc_data"]["sdof_parameters"] / f"{b}_sdof_parameters.json"
    with open(fp_sdof_params, "r") as file:
        sdof_params = json.load(file)

    fp_optimisation_params = cfg["proc_data"]["sdof_optimisation"] / f"{b}_optimised_parameters.json"    
    with open(fp_optimisation_params, "r") as file:
        hyst_params = json.load(file)
    
    ops_updates = {
        k:v for k,v in (sdof_params|hyst_params).items()
    }

    copy_structural_model(src_model, dst_model, ops_updates=ops_updates)

    # now copy the config 
    src_config = cfg["templates"]["config_ida_htf"]
    dst_config = folder / "config_ida_htf_femap695_set.py"

    changes = {
        "results_folder_name": "ida_femap695",
        "record_filenames": ['fema_p695_120111.json',
                             'fema_p695_120121.json',
                             'fema_p695_120411.json',
                             'fema_p695_120521.json',
                             'fema_p695_120611.json',
                             'fema_p695_120621.json',
                             'fema_p695_120711.json',
                             'fema_p695_120721.json',
                             'fema_p695_120811.json',
                             'fema_p695_120821.json',
                             'fema_p695_120911.json',
                             'fema_p695_120921.json',
                             'fema_p695_121011.json',
                             'fema_p695_121021.json',
                             'fema_p695_121111.json',
                             'fema_p695_121211.json',
                             'fema_p695_121221.json',
                             'fema_p695_121321.json',
                             'fema_p695_121411.json',
                             'fema_p695_121421.json',
                             'fema_p695_121511.json',
                             'fema_p695_121711.json',
                             ],
    }

    copy_analysis_config(src_config, dst_config, **changes)
    
    batch.append({
        "script": dst_run,
        "config": [dst_config]
        })
    
batch_run_src = cfg["templates"]["batch_run"]
batch_run_dst = cfg["scripts"]["wp1pt4pt1_batch_run"] / "sdof_3s_ida_femaP695.py"
configure_batch_run_file(batch_run_src, batch_run_dst, batch)

Create some files for running a full IDA on the approximate SDOF Systems

In [ ]:
batch = []

for b in buildings:
    # make the target folder
    folder = ROOT / f"{b}_appx_sdof"
    folder.mkdir(parents=True, exist_ok=True)

    # copy files that don't need changing:
    src_proc = cfg["templates"]["ida_process_recorder_x_displacement"]
    dst_proc = folder / "ida_process_recorders.py"
    copy_file(src_proc, dst_proc)

    src_im = cfg["templates"]["config_im_SA"]
    dst_im = folder / "config_im.py"
    copy_file(src_im, dst_im)

    src_inj = cfg["templates"]["nltha_injection_update_damping"]
    dst_inj = folder / "injection_functions.py"
    copy_file(src_inj, dst_inj)

    src_run = cfg["templates"]["run_ida_htf_multiple_records"]
    dst_run = folder / "run_ida_htf_multiple_records.py"
    copy_file(src_run, dst_run)

    # now copy the structural model
    # model already exists

    # now copy the config 
    src_config = cfg["templates"]["config_ida_htf"]
    dst_config = folder / "config_ida_htf_femap695_set.py"

    changes = {
        "results_folder_name": "ida_femap695",
        "record_filenames": ['fema_p695_120111.json',
                             'fema_p695_120121.json',
                             'fema_p695_120411.json',
                             'fema_p695_120521.json',
                             'fema_p695_120611.json',
                             'fema_p695_120621.json',
                             'fema_p695_120711.json',
                             'fema_p695_120721.json',
                             'fema_p695_120811.json',
                             'fema_p695_120821.json',
                             'fema_p695_120911.json',
                             'fema_p695_120921.json',
                             'fema_p695_121011.json',
                             'fema_p695_121021.json',
                             'fema_p695_121111.json',
                             'fema_p695_121211.json',
                             'fema_p695_121221.json',
                             'fema_p695_121321.json',
                             'fema_p695_121411.json',
                             'fema_p695_121421.json',
                             'fema_p695_121511.json',
                             'fema_p695_121711.json',
                             ],
    }

    copy_analysis_config(src_config, dst_config, **changes)
    
    batch.append({
        "script": dst_run,
        "config": [dst_config]
        })
    
batch_run_src = cfg["templates"]["batch_run"]
batch_run_dst = cfg["scripts"]["wp1pt4pt1_batch_run"] / "approx_sdof_3s_ida_femaP695.py"
configure_batch_run_file(batch_run_src, batch_run_dst, batch)

### Modal analyses of the SDOF systems

Creates modal-analysis files for the SDOF and approximate-SDOF systems so their fundamental
periods can be extracted (the MDOFs already have a `modal/` folder from `012`; the SDOF
`modal/` folders are empty). Uses `n_modes = 1` and the `-fullGenLapack` dense solver, which
is robust for these small systems.

In [ ]:
# Modal analysis of the SDOF systems -> populates <model>/modal/modal_properties.json
# (eigenPeriod). The MDOFs already have this; here we add it for the _sdof and _appx_sdof
# models. n_modes=1, solver="-fullGenLapack" (robust dense solver for these small systems).
sdof_modal_batch = []
appx_sdof_modal_batch = []

for b in buildings:
    for model_folder_name, batch in ((f"{b}_sdof", sdof_modal_batch),
                                     (f"{b}_appx_sdof", appx_sdof_modal_batch)):
        folder = ROOT / model_folder_name

        # run script (structural_model.py with model_init already exists in the folder)
        src_run = cfg["templates"]["run_modal"]
        dst_run = folder / "run_modal.py"
        copy_file(src_run, dst_run)

        # modal config: 1 mode, full dense eigen solver
        src_config = cfg["templates"]["config_modal"]
        dst_config = folder / "config_modal.py"
        copy_analysis_config(
            src_config,
            dst_config,
            results_folder_name="modal",
            update_config={"n_modes": 1, "solver": "-fullGenLapack"},
        )

        batch.append({"script": dst_run, "config": [dst_config]})

batch_run_src = cfg["templates"]["batch_run"]

configure_batch_run_file(
    batch_run_src,
    cfg["scripts"]["wp1pt4pt1_batch_run"] / "sdof_3s_modal.py",
    sdof_modal_batch,
)
configure_batch_run_file(
    batch_run_src,
    cfg["scripts"]["wp1pt4pt1_batch_run"] / "approx_sdof_3s_modal.py",
    appx_sdof_modal_batch,
)

---

# ⛔ RUN BARRIER — run the IDAs now

From `cfg["scripts"]["wp1pt4pt1_batch_run"]`:

| Launcher | Expect |
|---|---|
| `sdof_3s_modal.py` | seconds |
| `approx_sdof_3s_modal.py` | seconds |
| `sdof_3s_ida_femaP695.py` | slow |
| `approx_sdof_3s_ida_femaP695.py` | slow |
| `mdof_3s_ida_femaP695.py` | **very slow — of the order of 18–20 hours** |

Start the MDOF IDA first if you are running overnight; the two SDOF IDAs and the modal
analyses can be queued behind it (the process semaphore keeps the machine from being
oversubscribed).

`mdof_3s_idas_120111.py` is optional — a single-record IDA used only as a quick sanity check.

Results land in `<model_folder>/ida_femap695/` as `ida_results.pickle`,
`collapse_fragility.json` and the `fractile_{16,50,84}pc_ida_spline.csv` files.

**Do not continue past this point until these have finished.**

## §2 Collapse fragility curves

### 2.7.1 Fragility Curves

In [ ]:
total_n = len(buildings)
n_rows = int(round(total_n / 3, 0))
n_cols = 3
fig, axs = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3, ), sharex=True)

g = 9810        # mm/s^2
sdof_fragilities = {}
appx_sdof_fragilities = {}
mdof_fragilities = {}

for ax, b in zip(axs.flatten(), buildings):
    sdof_folder = ROOT / f"{b}_sdof" / "ida_femap695"
    appx_sdof_folder = ROOT / f"{b}_appx_sdof" / "ida_femap695"
    mdof_folder = ROOT / b / "ida_femap695"
    
    # load collapse fragility
    with open(sdof_folder / "collapse_fragility.json", "r") as file:
        sdof_cf_data = json.load(file) 

    sdof_fragilities[b] = sdof_cf_data
    x = np.linspace(0.01, 10 * g, 200) # in mm/s²
    sdof_y = lognorm.cdf(x, s=sdof_cf_data["dispersion"], scale=sdof_cf_data["median"])
    sdof_fragilities[b]["fc"] = np.column_stack([x, sdof_y])

    with open(appx_sdof_folder / "collapse_fragility.json", "r") as file:
        appx_sdof_cf_data = json.load(file) 

    appx_sdof_fragilities[b] = appx_sdof_cf_data
    appx_sdof_y = lognorm.cdf(x, s=appx_sdof_cf_data["dispersion"], scale=appx_sdof_cf_data["median"])
    appx_sdof_fragilities[b]["fc"] = np.column_stack([x, appx_sdof_y])

    try:
        with open(mdof_folder / "collapse_fragility.json", "r") as file:
            mdof_cf_data = json.load(file) 

        mdof_fragilities[b] = mdof_cf_data
        mdof_y = lognorm.cdf(x, s=mdof_cf_data["dispersion"], scale=mdof_cf_data["median"])
        mdof_fragilities[b]["fc"] = np.column_stack([x, mdof_y])

        # plot collapse fragility
        ax.plot(np.array(mdof_cf_data["efc"][0]) / g, mdof_cf_data["efc"][1], marker=".", ls="", color="k", label="MDOF ecdf")
        ax.plot(x / g, mdof_y, ls="-.", color="k", label="MDOF Fit")
    except FileNotFoundError:
        continue

    ax.plot(np.array(sdof_cf_data["efc"][0]) * gammas[b] / g, sdof_cf_data["efc"][1], marker=".", ls="", color="b", label="Eq. MDOF ecdf (opt.)")
    ax.plot(x * gammas[b] / g, sdof_y, ls="-.", color="b", label="Eq. MDOF Fit (opt.)")
    
    ax.plot(np.array(appx_sdof_cf_data["efc"][0]) * gammas[b] / g, appx_sdof_cf_data["efc"][1], marker=".", ls="", color="g", label="Eq. MDOF ecdf (appx.)")
    ax.plot(x * gammas[b] / g, appx_sdof_y, ls="-.", color="g", label="Eq. MDOF Fit (appx.)")

    ax.grid(which="both", ls="-.", color="0.8")
    ax.minorticks_on()
    ax.set_title(b)   

    for ax in axs.flatten()[-3:]:
        ax.set_xlabel("SA(T1) [g]")

    for ax in axs[:, 0]:
        ax.set_ylabel("Probability of Collapse")

    leg = axs[0, 0].legend()
    frame = leg.get_frame()
    frame.set_edgecolor("k")
    plt.tight_layout()

Collapse fragilities re-expressed in terms of **$AvgSA_{0-3s}$**. For each record the IDA scale factor that causes collapse (obtained in SA(T1)) is applied to the record's $AvgSA_{0-3s}$ to get the $AvgSA_{0-3s}$ level that causes collapse, then a lognormal collapse fragility is fit. $AvgSA_{0-3s}$ uses the same definition as the record selection (`standes.intensitymeasures.avgsa_03`). The eq-SDOF curves are mapped to MDOF-equivalent coordinates with `gammas[b]`, as for the SA(T1) fragilities above.

In [ ]:
# ---- Collapse fragilities re-expressed in terms of AvgSA(0-3s) ----
import re
from standes.intensitymeasures import SpectralAcceleration, avgsa_03
from standes.analysis.ida import get_collapse_iml, collapse_fragility_from_imls
from standes.groundmotion import load_ground_motion_from_json

g = 9810  # mm/s^2

# Map eq-SDOF AvgSA into MDOF-equivalent coordinates (same gammas[b] used for the
# SA(T1) fragilities above). A constant per-building scalar passes through the AvgSA
# geometric mean unchanged, so the same factor applies to AvgSA_03 as to SA(T1).
# Set False to see the raw eq-SDOF collapse AvgSA.
APPLY_GAMMA = True

_period_re = re.compile(r"period=([0-9.eE+-]+)")
_gm_cache = {}
_avgsa03_cache = {}   # AvgSA_03 (mm/s^2) is a record property -> compute once per record

def _gm_record(path):
    if path not in _gm_cache:
        _gm_cache[path] = load_ground_motion_from_json(path)
    return _gm_cache[path]

def _record_avgsa03(path):
    if path not in _avgsa03_cache:
        im = avgsa_03()
        im.set_iml(_gm_record(path), g)
        _avgsa03_cache[path] = im.iml
    return _avgsa03_cache[path]

def avgsa03_collapse_fragility(model_folder):
    """Collapse fragility (FragilityCurve) in AvgSA_03 for one model's IDA results."""
    ida_folder = model_folder / "ida_femap695"
    ida_results = pickle.load(open(ida_folder / "ida_results.pickle", "rb"))
    record_logs = json.load(open(ida_folder / "record_logs.json"))
    tags = [k for k in ida_results if k in record_logs and str(k).isdigit()]

    T1 = float(_period_re.search(record_logs[tags[0]]["intensity_measure"]).group(1))
    sa_im = SpectralAcceleration(T1)

    sas = []
    collapse_sas = []
    collapse_avgsa = []
    sfs = []
    
    for t in tags:
        path = record_logs[t]["record_path"]
        sa_im.set_iml(_gm_record(path), g)                       # unscaled SA(T1) of the record
        collapse_sa = get_collapse_iml(ida_results[t])
        scale_factor = collapse_sa / sa_im.iml
        sfs.append(scale_factor)
        sas.append(sa_im.iml)
        collapse_sas.append(collapse_sa)
        collapse_avgsa.append(scale_factor * _record_avgsa03(path))
    return (collapse_fragility_from_imls(collapse_avgsa), np.array(sas), 
            np.array(collapse_sas), np.array(sfs), np.array(collapse_avgsa))

avgsa03_mdof_fragilities = {}
avgsa_mdof_collapse_avgsa = {}
mdof_sas = {}
mdof_collapse_sas = {}
mdof_sfs = {}


avgsa03_sdof_fragilities = {}
avgsa03_appx_sdof_fragilities = {}

total_n = len(buildings)
n_rows = int(round(total_n / 3, 0))
n_cols = 3
fig, axs = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3), sharex=True)

x = np.linspace(0.01, 1.5 * g, 300)  # mm/s^2

for ax, b in zip(axs.flatten(), buildings):
    gamma = gammas[b] if APPLY_GAMMA else 1.0

    # MDOF
    try:
        mdof_frag, sas, collapse_sas, sfs, collapse_avgsa = avgsa03_collapse_fragility(ROOT / b)
    except FileNotFoundError:
        continue
    avgsa03_mdof_fragilities[b] = mdof_frag
    avgsa_mdof_collapse_avgsa[b] = collapse_avgsa
    mdof_sas[b] = sas
    mdof_collapse_sas[b] = collapse_sas
    mdof_sfs[b] = sfs

    mdof_y = lognorm.cdf(x, s=mdof_frag.dispersion, scale=mdof_frag.median)
    ax.plot(np.array(mdof_frag.efc[0]) / g, mdof_frag.efc[1], marker=".", ls="", color="k", label="MDOF ecdf")
    ax.plot(x / g, mdof_y, ls="-.", color="k", label="MDOF Fit")

    # optimised SDOF
    sdof_frag, *_ = avgsa03_collapse_fragility(ROOT / f"{b}_sdof")
    avgsa03_sdof_fragilities[b] = sdof_frag
    sdof_y = lognorm.cdf(x, s=sdof_frag.dispersion, scale=sdof_frag.median)
    ax.plot(np.array(sdof_frag.efc[0]) * gamma / g, sdof_frag.efc[1], marker=".", ls="", color="b", label="Eq. MDOF ecdf (opt.)")
    ax.plot(x * gamma / g, sdof_y, ls="-.", color="b", label="Eq. MDOF Fit (opt.)")

    # approximate SDOF
    appx_frag, *_ = avgsa03_collapse_fragility(ROOT / f"{b}_appx_sdof")
    avgsa03_appx_sdof_fragilities[b] = appx_frag
    appx_y = lognorm.cdf(x, s=appx_frag.dispersion, scale=appx_frag.median)
    ax.plot(np.array(appx_frag.efc[0]) * gamma / g, appx_frag.efc[1], marker=".", ls="", color="g", label="Eq. MDOF ecdf (appx.)")
    ax.plot(x * gamma / g, appx_y, ls="-.", color="g", label="Eq. MDOF Fit (appx.)")

    ax.grid(which="both", ls="-.", color="0.8")
    ax.minorticks_on()
    ax.set_title(b)

for ax in axs.flatten()[-3:]:
    ax.set_xlabel("AvgSA$_{0-3s}$ [g]")
for ax in axs[:, 0]:
    ax.set_ylabel("Probability of Collapse")

leg = axs[0, 0].legend()
leg.get_frame().set_edgecolor("k")
plt.tight_layout()

# save all the fragility curves to a pickle file
fp = cfg["proc_data"]["sdof_fragility_curves"] / "mdof_fragility_curves.pickle"
with open(fp, "wb") as file:
    pickle.dump(avgsa03_mdof_fragilities, file)

fp = cfg["proc_data"]["sdof_fragility_curves"] / "sdof_fragility_curves.pickle"
with open(fp, "wb") as file:
    pickle.dump(avgsa03_sdof_fragilities, file)

fp = cfg["proc_data"]["sdof_fragility_curves"] / "appx_sdof_fragility_curves.pickle"
with open(fp, "wb") as file:
    pickle.dump(avgsa03_appx_sdof_fragilities, file)


### Periods of the three model types

Periods of the MDOF, optimised-SDOF and approximate-SDOF models for each building, so any
differences can be seen. Two sources per system: the period actually used in the IDA (parsed
from the `SpectralAcceleration` entry in `record_logs.json`) and the eigenperiod from the
modal analysis (`modal/modal_properties.json`).

In [ ]:
# ---- Periods of the MDOF, SDOF and approximate-SDOF models per building ----
_period_re = re.compile(r"period=([0-9.eE+-]+)")

def _T_ida(model_folder):
    rl = json.load(open(model_folder / "ida_femap695" / "record_logs.json"))
    tag = [k for k in rl if str(k).isdigit()][0]
    return float(_period_re.search(rl[tag]["intensity_measure"]).group(1))

def _T_modal(model_folder):
    props = json.load(open(model_folder / "modal" / "modal_properties.json"))
    return float(props["eigenPeriod"][0])

period_rows = {}
for b in buildings:
    period_rows[b] = {
        "T_mdof (IDA)": _T_ida(ROOT / b),
        "T_mdof (modal)": _T_modal(ROOT / b),
        "T_sdof (IDA)": _T_ida(ROOT / f"{b}_sdof"),
        "T_sdof (modal)": _T_modal(ROOT / f"{b}_sdof"),
        "T_appx_sdof (IDA)": _T_ida(ROOT / f"{b}_appx_sdof"),
        "T_appx_sdof (modal)": _T_modal(ROOT / f"{b}_appx_sdof"),
    }

periods_df = pd.DataFrame(period_rows).T[[
    "T_mdof (IDA)", "T_mdof (modal)",
    "T_sdof (IDA)", "T_sdof (modal)",
    "T_appx_sdof (IDA)", "T_appx_sdof (modal)",
]].round(4)
periods_df

### Comparison of dispersion and median values

In [ ]:
fig, ax = plt.subplots()
ax_twin = ax.twinx()

theta_ratios = []
beta_ratios = []
appx_theta_ratios = []
appx_beta_ratios = []
avgSA_theta_ratios = []
avgSA_beta_ratios = []
avgSA_appx_theta_ratios = []
avgSA_appx_beta_ratios = []
tick_labels = []

for b in buildings:
    sdof_theta = sdof_fragilities[b]["median"] * gammas[b]
    sdof_beta = sdof_fragilities[b]["dispersion"]

    avgSA_sdof_theta = avgsa03_sdof_fragilities[b].median * gammas[b]
    avgSA_sdof_beta = avgsa03_sdof_fragilities[b].dispersion

    appx_sdof_theta = appx_sdof_fragilities[b]["median"] * gammas[b]
    appx_sdof_beta = appx_sdof_fragilities[b]["dispersion"]

    avgSA_appx_sdof_theta = avgsa03_sdof_fragilities[b].median * gammas[b]
    avgSA_appx_sdof_beta = avgsa03_sdof_fragilities[b].dispersion

    try:
        mdof_theta = mdof_fragilities[b]["median"]
        mdof_beta = mdof_fragilities[b]["dispersion"]

        avgSA_mdof_theta = avgsa03_sdof_fragilities[b].median
        avgSA_mdof_beta = avgsa03_sdof_fragilities[b].dispersion

        theta_ratios.append(mdof_theta / sdof_theta)
        beta_ratios.append(mdof_beta / sdof_beta)

        appx_theta_ratios.append(mdof_theta / appx_sdof_theta)
        appx_beta_ratios.append(mdof_beta / appx_sdof_beta)

        avgSA_theta_ratios.append(avgSA_mdof_theta / avgSA_sdof_theta)
        avgSA_beta_ratios.append(avgSA_mdof_beta / avgSA_sdof_beta)

        avgSA_appx_theta_ratios.append(avgSA_mdof_theta / avgSA_appx_sdof_theta)
        avgSA_appx_beta_ratios.append(avgSA_mdof_beta / avgSA_appx_sdof_beta)

    except KeyError:
        continue

    tick_labels.append(b)
    print(mdof_theta/9810, sdof_theta/9810, theta_ratios[-1], mdof_beta, sdof_beta, beta_ratios[-1])

ax.grid(ls="-.", color="0.8")
ax.hlines(1.0, -1, 9, color="k", ls="-.")
ax.plot(theta_ratios, ls="", marker="o", mec="k", mfc="b", label=r"$\theta_{MDOF}$/$\theta_{SDOF}$ (opti.)")
ax_twin.plot(beta_ratios, ls="", marker="s", mec="k", mfc="b", label=r"$\beta_{MDOF}$/$\beta_{SDOF}$ (opti.)")
ax.plot(appx_theta_ratios, ls="", marker="o", mec="k", mfc="r", label=r"$\theta_{MDOF}$/$\theta_{SDOF}$ (appx.)")
ax_twin.plot(appx_beta_ratios, ls="", marker="s", mec="k", mfc="r", label=r"$\beta_{MDOF}$/$\beta_{SDOF}$ (appx.)")
ax.plot(avgSA_theta_ratios, ls="", marker="o", mec="k", mfc="g", label=r"$\theta_{MDOF}$/$\theta_{SDOF}$ (AvgSA$_{0-3s}$)")
ax_twin.plot(avgSA_beta_ratios, ls="", marker="s", mec="k", mfc="g", label=r"$\beta_{MDOF}$/$\beta_{SDOF}$ (AvgSA$_{0-3s}$)")
ax.plot(avgSA_appx_theta_ratios, ls="", marker="o", mec="k", mfc="c", label=r"$\theta_{MDOF}$/$\theta_{SDOF}$ (AvgSA$_{0-3s}$, appx.)")
ax_twin.plot(avgSA_appx_beta_ratios, ls="", marker="s", mec="k", mfc="c", label=r"$\beta_{MDOF}$/$\beta_{SDOF}$ (AvgSA$_{0-3s}$, appx.)")

ax.fill_between([-1, 9], 1.6, 1.0, color="green", alpha=0.15)
ax.fill_between([-1, 9], 0.6, 1.0, color="red", alpha=0.15)

ax.set_xlim(-0.5, 8.5)
ax.set_ylim(0.6, 1.6)
ax.set_ylabel(r"Median Capacity Ratio: $\theta_{MDOF}$/$\theta_{SDOF}$")
ax.set_xticks(range(len(tick_labels)))
ax.set_xticklabels(tick_labels, rotation=60, ha="right")


ax_twin.set_ylim(0.6, 1.6)
ax_twin.set_ylabel(r"Dispersion Ratio: $\beta_{MDOF}$/$\beta_{SDOF}$")

legend_handles = ax.get_legend_handles_labels()[0] + ax_twin.get_legend_handles_labels()[0]
legend_labels = ax.get_legend_handles_labels()[1] + ax_twin.get_legend_handles_labels()[1]
leg = ax.legend(legend_handles, legend_labels, bbox_to_anchor=(1.15, 1), loc="upper left", borderaxespad=0., frameon=True)
frame = leg.get_frame()
frame.set_edgecolor("k")



## §3 Median IDA curves

In [ ]:
total_n = len(buildings)
n_rows = int(round(total_n / 3, 0))
n_cols = 3
fig, axs = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3, ), sharex=True, sharey=True)

g = 9810        # mm/s^2
height = 10500  # mm
x_max = 1000

for ax, b in zip(axs.flatten(), buildings):
    sdof_folder = ROOT / f"{b}_sdof" / "ida_femap695" /"roof_displacement/ida_splines"
    appx_sdof_folder = ROOT / f"{b}_appx_sdof" / "ida_femap695" /"roof_displacement/ida_splines"
    mdof_folder = ROOT / b / "ida_femap695" /"roof_drift/ida_splines"
    
    # load the fractile splines
    sdof_pc16 = np.loadtxt(sdof_folder / "fractile_16pc_ida_spline.csv", delimiter=",")
    sdof_pc50 = np.loadtxt(sdof_folder / "fractile_50pc_ida_spline.csv", delimiter=",")
    sdof_pc84 = np.loadtxt(sdof_folder / "fractile_84pc_ida_spline.csv", delimiter=",")

    sdof_pc16 = np.concatenate([sdof_pc16, [[x_max, sdof_pc16[-1, 1]]]])
    sdof_pc50 = np.concatenate([sdof_pc50, [[x_max, sdof_pc50[-1, 1]]]])
    sdof_pc84 = np.concatenate([sdof_pc84, [[x_max, sdof_pc84[-1, 1]]]])

    appx_sdof_pc16 = np.loadtxt(appx_sdof_folder / "fractile_16pc_ida_spline.csv", delimiter=",")
    appx_sdof_pc50 = np.loadtxt(appx_sdof_folder / "fractile_50pc_ida_spline.csv", delimiter=",")
    appx_sdof_pc84 = np.loadtxt(appx_sdof_folder / "fractile_84pc_ida_spline.csv", delimiter=",")

    appx_sdof_pc16 = np.concatenate([appx_sdof_pc16, [[x_max, appx_sdof_pc16[-1, 1]]]])
    appx_sdof_pc50 = np.concatenate([appx_sdof_pc50, [[x_max, appx_sdof_pc50[-1, 1]]]])
    appx_sdof_pc84 = np.concatenate([appx_sdof_pc84, [[x_max, appx_sdof_pc84[-1, 1]]]])

    try:
        mdof_pc16 = np.loadtxt(mdof_folder / "fractile_16pc_ida_spline.csv", delimiter=",")
        mdof_pc50 = np.loadtxt(mdof_folder / "fractile_50pc_ida_spline.csv", delimiter=",")
        mdof_pc84 = np.loadtxt(mdof_folder / "fractile_84pc_ida_spline.csv", delimiter=",")

        mdof_pc16 = np.concatenate([mdof_pc16, [[x_max / height, mdof_pc16[-1, 1]]]])
        mdof_pc50 = np.concatenate([mdof_pc50, [[x_max / height, mdof_pc50[-1, 1]]]])
        mdof_pc84 = np.concatenate([mdof_pc84, [[x_max / height, mdof_pc84[-1, 1]]]])
    except FileNotFoundError:
         continue

    ax.plot(mdof_pc16[:, 0] * height, mdof_pc16[:, 1] / g, ls="-.", color="k", label=r"16% MDOF", alpha=1.0, lw=1.5)
    ax.plot(mdof_pc50[:, 0] * height, mdof_pc50[:, 1] / g, ls="-", color="k", label=r"50% MDOF", lw=2.5)
    ax.plot(mdof_pc84[:, 0] * height, mdof_pc84[:, 1] / g, ls="-.", color="k", label=r"84% MDOF", alpha=1.0, lw=1.5)
    ax.plot(sdof_pc16[:, 0] * gammas[b], sdof_pc16[:, 1] * gammas[b] / g, ls="-.", color="b", label=r"16% Eq. MDOF", alpha=1, lw=1.5)
    ax.plot(sdof_pc50[:, 0] * gammas[b], sdof_pc50[:, 1] * gammas[b] / g, ls="-", color="b", label=r"50% Eq. MDOF", lw=2.5)
    ax.plot(sdof_pc84[:, 0] * gammas[b], sdof_pc84[:, 1] * gammas[b] / g, ls="-.", color="b", label=r"84% Eq. MDOF", alpha=1, lw=1.5)
    ax.plot(appx_sdof_pc16[:, 0] * gammas[b], appx_sdof_pc16[:, 1] * gammas[b] / g, ls="-.", color="g", label=r"16% Eq. MDOF", alpha=1, lw=1.5)
    ax.plot(appx_sdof_pc50[:, 0] * gammas[b], appx_sdof_pc50[:, 1] * gammas[b] / g, ls="-", color="g", label=r"50% Eq. MDOF", lw=2.5)
    ax.plot(appx_sdof_pc84[:, 0] * gammas[b], appx_sdof_pc84[:, 1] * gammas[b] / g, ls="-.", color="g", label=r"84% Eq. MDOF", alpha=1, lw=1.5)
    
    
    
    ax.grid(which="major", ls="-.", color="0.8")
    ax.set_title(b)
    # ax.minorticks_on()

ax.set_xlim(0, x_max)
ax.set_ylim(0, 6)
leg = axs.flatten()[0].legend()
frame = leg.get_frame()
frame.set_edgecolor("k")
plt.tight_layout()

for ax in axs.flatten()[-3:]:
        ax.set_xlabel("Roof Displacement [mm]")

for ax in axs[:, 0]:
    ax.set_ylabel("SA(T1) [g]")

## §4 Combined backbones

How the total (frame + brace) predicted backbone varies with the design base-shear
coefficient and with the braced-bay aspect ratio. `total_backbone` is imported from
`phd_project/scripts/sdof_parameterisation.py`.

In [ ]:
(bdata_df["seismic_mass"] * 9.81)

### Variation with $V_b$

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12,3), sharey=True)

Wt = 3060000
hi = 3500
aspect = 0.5
vbs = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7])

cmap = plt.get_cmap("viridis")
norm = plt.Normalize(vmin=vbs.min(), vmax=vbs.max())

for vb in vbs:
    bb_br = get_approximate_br_backbone(vb, Wt, hi, aspect)
    bb_ss = get_approximate_ss_backbone(vb, Wt, hi)
    bb_t = total_backbone(vb, Wt, hi, aspect)
    color = cmap(norm(vb))

    axs[0].plot(bb_ss[:, 0], bb_ss[:, 1] / 1000, color=color, lw=2)
    axs[1].plot(bb_br[:, 0], bb_br[:, 1] / 1000, color=color, lw=2)
    axs[2].plot(bb_t[:, 0], bb_t[:, 1] / 1000, color=color, lw=2)

for ax in axs:
    ax.set_xlabel("SDOF Displacement [mm]")
    ax.set_xlim(0, 500)
    ax.grid(ls="-.", color="0.8")

axs[0].set_ylabel("SDOF Force [kN]")
axs[0].set_title("Soft Storey Mech.")
axs[1].set_title("Brace Contr.")
axs[2].set_title("Total")
ax.set_ylim(0)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
cbar = fig.colorbar(sm, ax=axs[2])
cbar.set_label("$C_d$", rotation=90, labelpad=15)

plt.tight_layout()

### Variation with aspect ratio

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12,3), sharey=True)

Wt = 3060000
hi = 3500
vb = 0.25
aspects = np.array([0.5, 0.6, 0.75, 0.8, 1.0, 1.2])

cmap = plt.get_cmap("viridis")
norm = plt.Normalize(vmin=aspects.min(), vmax=aspects.max())

for aspect in aspects:
    bb_br = get_approximate_br_backbone(vb, Wt, hi, aspect)
    bb_ss = get_approximate_ss_backbone(vb, Wt, hi)
    bb_t = total_backbone(vb, Wt, hi, aspect)
    color = cmap(norm(aspect))

    axs[0].plot(bb_ss[:, 0], bb_ss[:, 1] / 1000, color=color, lw=2)
    axs[1].plot(bb_br[:, 0], bb_br[:, 1] / 1000, color=color, lw=2)
    axs[2].plot(bb_t[:, 0], bb_t[:, 1] / 1000, color=color, lw=2)

for ax in axs:
    ax.set_xlabel("SDOF Displacement [mm]")
    ax.set_xlim(0, 500)
    ax.grid(ls="-.", color="0.8")

axs[0].set_ylabel("SDOF Force [kN]")
axs[0].set_title("Soft Storey Mech.")
axs[1].set_title("Brace Contr.")
axs[2].set_title("Total")
ax.set_ylim(0)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
cbar = fig.colorbar(sm, ax=axs[2])
cbar.set_label("$Brace Aspect Ratio$", rotation=90, labelpad=15)

plt.tight_layout()

---

## Outputs and downstream use

Written to `cfg["proc_data"]["sdof_fragility_curves"]`:

| File | Consumed by |
|---|---|
| `mdof_fragility_curves.pickle` | `017-disagg_imls_for_msa_stripes` |
| `sdof_fragility_curves.pickle` | `017-disagg_imls_for_msa_stripes` |
| `appx_sdof_fragility_curves.pickle` | comparison only |

`017` combines these with the site hazard curves from `004` to pick the intensity levels for
the multiple-stripe analysis; `050_setup_msa_runs_for_complete_sites` then adds MSA files to
the model folders built by `012` and `013`.